In [1]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
from datetime import datetime
import os
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

c:\Users\Leo\.conda\envs\tensor4\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
data = pd.read_csv(r'..\data\train.csv')
test = pd.read_csv(r'..\data\test.csv')

sp_cols = ['RoomService','FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def feat_eng(df, train=False):
    df['VIP'] = df['VIP'].astype(bool)
    df[['FN', 'LN']] = df['Name'].str.split(' ', expand=True)
    df['Family'] = df['LN'].duplicated(keep=False).map({True: 'family', False: 'not family'})
    df['sp_cols'] = (df[sp_cols] != 0).sum(axis=1)
    df[['deck', 'room', 'side']] = df['Cabin'].str.split('/', expand=True)
    
    # Define a nested function to calculate the rate
    try:
        vip = (df['VIP']==True).sum()
        transp = (df['sp_cols']==True).sum()

        df['Rate'] = np.where((df.get('VIP', 0) == True) & (df.get('Family', 0)=="family"),  
                            1 - (vip / transp), vip / transp)
    except:
        pass       

    numeric_transformer = Pipeline([
                ('imputer', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler())
            ])
            
    categorical_cols = ['HomePlanet','CryoSleep', 'Destination', 'VIP', 
                        'Family', 'deck', 'side'
                        ]
    
    num_cols = ['Age','RoomService', 'FoodCourt', 'ShoppingMall', 'Spa','VRDeck','sp_cols','room','Rate']

    # Create a preprocessing pipeline

    preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, num_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
            ])

    if train:
        X = df.drop(columns=['PassengerId','Name', 'FN','LN','Cabin','Transported'])
        y = df['Transported']

        # Split the data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        y_train = y_train.to_numpy()
        y_test = y_test.to_numpy()
        # Fit and transform the training data
        X_train_preprocessed = preprocessor.fit_transform(X_train)
        X_test_preprocessed = preprocessor.transform(X_test)

            # Create the DMatrix for XGBoost
        #dtrain = xgb.DMatrix(X_train_preprocessed, label=y_train)
        #dtest = xgb.DMatrix(X_test_preprocessed, label=y_test)

        return X_train_preprocessed, X_test_preprocessed, y_train, y_test

    else:
        X= df.drop(columns=['PassengerId','Name', 'FN','LN','Cabin',]) 
        X_blind = preprocessor.fit_transform(X)

        return X_blind
    
X_train, X_test, y_train, y_test = feat_eng(data, train=True)
blind = feat_eng(test, train=False)

In [14]:
X_train

array([[-0.0579459 , -0.3332135 , -0.25771591, ...,  0.        ,
         0.        ,  1.        ],
       [-0.82767227, -0.3332135 ,  0.4736393 , ...,  1.        ,
         0.        ,  0.        ],
       [-0.0579459 , -0.3332135 , -0.2930006 , ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.47779665, -0.09693741, -0.2930006 , ...,  1.        ,
         0.        ,  0.        ],
       [ 0.36190485,  0.23355005, -0.2930006 , ...,  1.        ,
         0.        ,  0.        ],
       [-0.0579459 , -0.32274557,  0.0207123 , ...,  1.        ,
         0.        ,  0.        ]])

In [13]:
blind

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Family,sp_cols,deck,room,side,Rate
0,Earth,True,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,family,0,G,3,S,0.605072
1,Earth,False,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,not family,2,F,4,S,0.605072
2,Europa,True,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,not family,0,C,0,S,0.605072
3,Europa,False,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,not family,3,C,1,S,0.605072
4,Earth,False,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,family,2,F,5,S,0.605072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,Earth,True,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,family,0,G,1496,S,0.605072
4273,Earth,False,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,family,4,NaN,NaN,NaN,0.605072
4274,Mars,True,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,family,0,D,296,P,0.605072
4275,Europa,False,NaN,NaN,False,0.0,2680.0,0.0,0.0,523.0,family,2,D,297,P,0.605072


In [4]:
# Initialize Optuna study
study = optuna.create_study(direction='maximize')

# Define the optimization function directly
def objective(trial):
    print(trial)
    params = {
        'objective': 'binary:logistic',
        'eval_metric': ['error', 'auc'],
        'tree_method': 'gpu_hist',
        #'enable_categorical': True,
        'device': 'cuda',
        'verbosity':0,
        
        # Parameters to optimize
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.3, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'seed': 42
    }
    
    # Cross validation
    cv_scores = []
    kf = StratifiedKFold(n_splits=5, 
                         shuffle=True, 
                         random_state=42)
    
    for train_idx, val_idx in kf.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    
    # Create DMatrix for XGBoost
        dtrain = xgb.DMatrix(X_fold_train, label=y_fold_train)
        dval = xgb.DMatrix(X_fold_val, label=y_fold_val)
        
        # Train model
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            early_stopping_rounds=20,
            evals=[(dval, 'val')],
            verbose_eval=False
        )
        
        # Get validation score
        val_pred = model.predict(dval)
        score = roc_auc_score(y_fold_val, val_pred)
        cv_scores.append(score)
    
    return np.mean(cv_scores)

# Run optimization
study.optimize(objective, n_trials=15, show_progress_bar=True)

# Get best parameters
best_params = study.best_params

# Add fixed parameters to best_params
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': ['error', 'auc'],
    'tree_method': 'gpu_hist',
    'device': 'cuda',
    'seed': 42
})

# Print results
print("\nBest parameters:", best_params)
print(f"Best CV score: {study.best_value:.4f}")



[I 2025-02-25 14:25:53,968] A new study created in memory with name: no-name-4dcf8e04-46fa-4ae9-9fe8-6f3d77e41b2b
c:\Users\Leo\.conda\envs\tensor4\lib\site-packages\optuna\progress_bar.py:49: ExperimentalWarning: Progress bar is experimental (supported from v1.2.0). The interface can change in the future.
  self._init_valid()
  0%|          | 0/15 [00:00<?, ?it/s]

  7%|▋         | 1/15 [00:06<01:26,  6.18s/it]

[I 2025-02-25 14:26:00,167] Trial 0 finished with value: 0.8951435157137165 and parameters: {'max_depth': 12, 'eta': 0.08611248529952709, 'subsample': 0.32098269912585425, 'colsample_bytree': 0.9435374847603215, 'min_child_weight': 3, 'lambda': 0.48436987349006105}. Best is trial 0 with value: 0.8951435157137165.


 13%|█▎        | 2/15 [00:15<01:44,  8.06s/it]

[I 2025-02-25 14:26:09,535] Trial 1 finished with value: 0.8956980988934932 and parameters: {'max_depth': 11, 'eta': 0.07104220895832311, 'subsample': 0.5642811817151847, 'colsample_bytree': 0.5204760301080986, 'min_child_weight': 2, 'lambda': 0.00010850821897108661}. Best is trial 1 with value: 0.8956980988934932.


...

[I 2025-02-25 14:27:54,763] Trial 12 finished with value: 0.9008913423999185 and parameters: {'max_depth': 7, 'eta': 0.029707034154878485, 'subsample': 0.9907550899537183, 'colsample_bytree': 0.7689820747112566, 'min_child_weight': 5, 'lambda': 0.001177771420126861}. Best is trial 12 with value: 0.9008913423999185.


 93%|█████████▎| 14/15 [02:04<00:07,  7.12s/it]

[I 2025-02-25 14:27:58,612] Trial 13 finished with value: 0.898012307305103 and parameters: {'max_depth': 8, 'eta': 0.1262539971837093, 'subsample': 0.9687902287950703, 'colsample_bytree': 0.7631284051606783, 'min_child_weight': 5, 'lambda': 0.0029652567020829563}. Best is trial 12 with value: 0.9008913423999185.


100%|██████████| 15/15 [02:12<00:00,  8.86s/it]

[I 2025-02-25 14:28:06,947] Trial 14 finished with value: 0.9013484304260937 and parameters: {'max_depth': 7, 'eta': 0.03446007715930608, 'subsample': 0.8364830685927181, 'colsample_bytree': 0.7285149864599313, 'min_child_weight': 5, 'lambda': 0.008382585006649011}. Best is trial 14 with value: 0.9013484304260937.

Best parameters: {'max_depth': 7, 'eta': 0.03446007715930608, 'subsample': 0.8364830685927181, 'colsample_bytree': 0.7285149864599313, 'min_child_weight': 5, 'lambda': 0.008382585006649011, 'objective': 'binary:logistic', 'eval_metric': ['error', 'auc'], 'tree_method': 'gpu_hist', 'device': 'cuda', 'seed': 42}
Best CV score: 0.9013


In [5]:
# Train final model with best parameters
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_test, label=y_test)

final_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=10000,
    early_stopping_rounds=50,
    evals=[(dval, 'validation')],
    verbose_eval=100
)

[0]	validation-error:0.25532	validation-auc:0.83769
[100]	validation-error:0.20184	validation-auc:0.89399
[200]	validation-error:0.19897	validation-auc:0.89873
[300]	validation-error:0.19437	validation-auc:0.89993
[348]	validation-error:0.19551	validation-auc:0.89938


In [10]:
blind

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Family,sp_cols,deck,room,side,Rate
0,Earth,True,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,family,0,G,3,S,0.605072
1,Earth,False,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,not family,2,F,4,S,0.605072
2,Europa,True,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,not family,0,C,0,S,0.605072
3,Europa,False,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,not family,3,C,1,S,0.605072
4,Earth,False,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,family,2,F,5,S,0.605072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,Earth,True,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,family,0,G,1496,S,0.605072
4273,Earth,False,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,family,4,NaN,NaN,NaN,0.605072
4274,Mars,True,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,family,0,D,296,P,0.605072
4275,Europa,False,NaN,NaN,False,0.0,2680.0,0.0,0.0,523.0,family,2,D,297,P,0.605072


In [16]:
dblind = xgb.DMatrix(blind)
y_pred_prob = final_model.predict(dblind)
# Calculate the accuracy
predicted_y  = (y_pred_prob  > 0.5).astype(int)

sub = pd.merge(test[['PassengerId']], pd.DataFrame(predicted_y), left_index=True, right_index=True)#.to_csv('data/ak_submission.csv', index=False)
sub = sub.rename(columns={0:'Transported'})
sub['Transported'] = sub['Transported'].map({0:False, 1:True})
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submissionxgboost_{timestamp}.csv"
filepath = os.path.join(r"../submissions", filename)  # Cross-platform

# Save DataFrame to CSV
sub.to_csv(filepath, index=False)